### TUGAS DATA MINING ASSOCIATION LEARNING
Jonathan Joyo Wibowo

5054251026

Rekayasa Kecerdasan Artifisial

### SETUP

In [69]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
from google.colab import drive

from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
SEED = 67  # six seven

In [70]:
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [71]:
uploaded = drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)

### LOAD DATA

In [73]:
df_data = pd.read_excel("/content/drive/MyDrive/dataset/online_retail/Online_Retail.xlsx")

In [74]:
df_data_raw = df_data.copy()

### EDA

In [75]:
df_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [76]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [77]:
df_data.nunique()

,0
InvoiceNo,25900
StockCode,4070
Description,4223
Quantity,722
InvoiceDate,23260
UnitPrice,1630
CustomerID,4372
Country,38


In [78]:
df_data.shape

(541909, 8)

In [79]:
df_data["Quantity"].describe()

,Quantity
count,541909.000000
mean,9.552250
std,218.081158
min,-80995.000000
25%,1.000000
50%,3.000000
75%,10.000000
max,80995.000000


In [80]:
df_data[df_data["Quantity"] < 1].count()

,0
InvoiceNo,10624
StockCode,10624
Description,9762
Quantity,10624
InvoiceDate,10624
UnitPrice,10624
CustomerID,8905
Country,10624


### Preprocessing

In [81]:
df_data = df_data_raw.copy()

In [82]:
# Hapus duplikat
df_data = df_data.drop_duplicates()

# Hapus duplikat pada subset InvoiceNo, StockCode
# Karena kita tidak peduli seberapa banyak suatu barang ada pada suatu basket/transaksi
# Kita hanya peduli apakah suatu barang ADA atau TIDAK

df_data = df_data.drop_duplicates(subset=["InvoiceNo", "StockCode"])

In [83]:
# Hapus data yang quantity <= 0
df_data = df_data[df_data["Quantity"] > 0]
df_data = df_data.reset_index(drop=True)

# Hapus juga yang quantity >= 80000 (tidak normal)
df_data = df_data[df_data["Quantity"] < 80000]
df_data = df_data.reset_index(drop=True)

In [84]:
df_data.shape

(520749, 8)

In [85]:
# Hapus transaksi yang di-cancel
cancelled_mask = df_data["InvoiceNo"].str.contains(r"^C", case=False, na=False)
df_data = df_data[~cancelled_mask]

In [86]:
df_data.shape

(520749, 8)

In [87]:
# drop kolom yang tidak perlu
# kita cuma pake InvoiceNo sama StockCode

# InvoiceNo buat ngelompokkin barang jadi basket-basket belanja

# StockCode buat mengidentifikasi barangnya
# sebenernya bisa pake Description. StockCode dan Description itu multikolinear,
# tapi karena Deskripsi lebih rawan typo, saya pake StockCode

df_data = df_data.drop(columns=["Description", "Quantity", "InvoiceDate",	"UnitPrice", "CustomerID", "Country"], axis=1)

In [88]:
df_data.head()

,InvoiceNo,StockCode
0,536365,85123A
1,536365,71053
2,536365,84406B
3,536365,84029G
4,536365,84029E


In [89]:
# Kelompokkan data ke dalam basket
basket = pd.crosstab(df_data["InvoiceNo"], df_data["StockCode"])


In [90]:
basket.head()

StockCode  10002  10080  10120  10125  10133  10135  11001  15030  15034  15036  15039  16008  16010  16011  16012  16014  16015  16016  16033  16043  16045  16046  16048  16049  16052  16054  16216  16218  16219  16225  16235  16236  16237  16238  16254  16259  17001  17003  17021  17038  17096  17174  18007  20615  20616  20617  20618  20619  20622  20652  20653  20654  20655  20657  20658  20659  20661  20662  20663  20664  20665  20666  20667  20668  20669  20670  20671  20674  20675  20676  20677  20678  20679  20681  20682  20684  20685  20686  20694  20695  20696  20697  20698  20699  20700  20701  20702  20703  20704  20705  20707  20711  20712  20713  20716  20717  20718  20719  20723  20724  20725  20726  20727  20728  20731  20733  20734  20735  20748  20749  20750  20751  20752  20754  20755  20756  20757  20758  20759  20760  20761  20762  20763  20764  20765  20766  20767  20768  20769  20770  20771  20772  20773  20774  20775  20777  20778  20780  20781  20782  20785  \
InvoiceNo                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
536365         0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0   
536366         0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0   


In [91]:
basket.describe()

StockCode         10002         10080         10120         10125         10133         10135         11001         15030         15034         15036         15039         16008         16010         16011         16012         16014         16015         16016         16033         16043         16045         16046         16048         16049         16052         16054         16216         16218         16219         16225         16235         16236         16237         16238         16254         16259         17001         17003         17021         17038         17096         17174         18007         20615         20616         20617         20618         20619         20622         20652         20653         20654         20655         20657         20658         20659         20661         20662         20663         20664         20665         20666         20667         20668         20669         20670         20671        20674         20675         20676  \
count      20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.000000  20727.00000  20727.000000  20727.000000   
mean           0.003474      0.001158      0.001399      0.004390      0.009456      0.008443      0.005452      0.000627      0.006658      0.024992      0.006996      0.003377      0.000145      0.002219      0.001689      0.002605      0.001206      0.003136      0.000241      0.000096      0.003281      0.000724      0.003232      0.000289      0.000627      0.001592      0.002171      0.003088      0.004825      0.004921      0.008105      0.006561      0.013991      0.006465      0.000820      0.001496      0.000048      0.011531      0.002268      0.001640      0.001158      0.000145      0.001447      0.004680      0.003425      0.002075      0.003667      0.004776      0.002461      0.003860      0.001592      0.002364      0.004776      0.001592      0.005066      0.002991      0.001592      0.000965      0.000965      0.000917      0.007575      0.000820      0.000772      0.010373      0.004632      0.000531      0.000096      0.00989      0.015535      0.021614   
std            0.058837      0.034009      0.037380      0.066116      0.096785      0.091500      0.073637      0.025037      0.081326      0.156103      0.083349      0.058017      0.012030      0.047059      0.041059      0.050977      0.034710      0.055914      0.015530      0.009823      0.057185      0.026892      0.056764      0.017012      0.025037      0.039871      0.046545      0.055483      0.069293      0.069980      0.089666      0.080739      0.117458      0.080147      0.028628      0.038645      0.006946      0.106763      0.047566      0.040469      0.034009      0.012030      0.038018      0.068251      0.058429      0.045501      0.060444      0.068948      0.049544      0.062008      0.039871      0.048565      0.068948      0.039871      0.070996      0.054612      0.039871      0.031049      0.031049      0.030264      0.086704      0.028628      0.027774      0.101320      0.067900      0.023032      0.009823      0.09896      0.123672      0.145424   
min            0.000000      0.0

In [92]:
# Opsional, buat dataframe yang isinya StockCode & Description
# nanti bisa dipake buat lookup kira-kira StockCode yang muncul
# di rule association itu produk apa

df_lookup = df_data_raw.copy()
df_lookup = df_lookup[["StockCode", "Description"]]

# drop duplikat
df_lookup = df_lookup.drop_duplicates()

# convert StockCode ke string
df_lookup["StockCode"] = df_lookup["StockCode"].astype(str)

### MODELING

In [93]:
freq_itemsets_apriori = apriori(basket, min_support=0.01, use_colnames=True, low_memory=True)

rules_apriori = association_rules(freq_itemsets_apriori,
                          metric="confidence",
                          min_threshold=0.5)

In [94]:
freq_itemsets_fpgrowth = fpgrowth(basket, min_support=0.01, use_colnames=True)

rules_fpgrowth = association_rules(freq_itemsets_fpgrowth,
                          metric="confidence",
                          min_threshold=0.5)

In [95]:
rules_fpgrowth.sort_values(by="lift", ascending=False).head(3)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
373,(22916),(22917),0.011434,0.011579,0.010662,0.932489,80.532120,1.0,0.010530,14.640985,0.999006,0.863281,0.931699,0.926661
374,(22917),(22916),0.011579,0.011434,0.010662,0.920833,80.532120,1.0,0.010530,12.487145,0.999152,0.863281,0.919918,0.926661
375,(22916),(22918),0.011434,0.011483,0.010276,0.898734,78.269173,1.0,0.010145,9.761609,0.998642,0.812977,0.897558,0.896846


In [96]:
rules_fpgrowth.sort_values(by="lift", ascending=False).head(3)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
373,(22916),(22917),0.011434,0.011579,0.010662,0.932489,80.532120,1.0,0.010530,14.640985,0.999006,0.863281,0.931699,0.926661
374,(22917),(22916),0.011579,0.011434,0.010662,0.920833,80.532120,1.0,0.010530,12.487145,0.999152,0.863281,0.919918,0.926661
375,(22916),(22918),0.011434,0.011483,0.010276,0.898734,78.269173,1.0,0.010145,9.761609,0.998642,0.812977,0.897558,0.896846


In [97]:
stock_codes = ["22916", "22917", "22918"]

df_lookup[df_lookup["StockCode"].isin(stock_codes)]

,StockCode,Description
974,22917,HERB MARKER ROSEMARY
977,22918,HERB MARKER PARSLEY
978,22916,HERB MARKER THYME


### INTERPRETASI

Ada 3 Aturan dengan skor lift tertinggi, yaitu:
1. Herb Marker Thyme (22916) -> Herb Marker Rosemary (22917)
2. Herb Marker Rosemary (22917) -> Herb Marker Thyme (22916)
3. Herb Marker Thyme (22916) -> Herb Marker Parsley (22918)

Aturan Pertama itu Confidencenya 93.24%, berarti jika Herb Marker Thyme (22916) dibeli, maka kemungkinan buat Herb Marker Rosemary (22917) dibeli itu 93.24%

Liftnya 80.53, berarti jika Herb Marker Thyme (22916) dibeli, maka kemungkinan buat Herb Marker Rosemary (22917) dibeli itu naik 80.53 kali lipat, yang awalnya 1.15% (Antecedent Consequent), naik jadi 93.24%

Aturan Kedua itu Confidencenya 92.08%, berarti jika Herb Marker Rosemary (22917) dibeli, maka kemungkinan buat Herb Marker Thyme (22916) dibeli itu 92.08%

Liftnya 80.53, berarti jika Herb Marker Rosemary (22917) dibeli, maka kemungkinan buat Herb Marker Thyme (22916) dibeli itu naik 80.53 kali lipat, yang awalnya 1.14% (Antecedent Consequent), naik jadi 92.08%

Aturan Pertama itu Confidencenya 89.87%, berarti jika Herb Marker Thyme (22916) dibeli, maka kemungkinan buat Herb Marker Parsley (22918) dibeli itu 89.87%

Liftnya 78.26, berarti jika Herb Marker Thyme (22916) dibeli, maka kemungkinan buat Herb Marker Parsley (22918) dibeli itu naik 78.26 kali lipat, yang awalnya 1.14% (Antecedent Consequent), naik jadi 89.87%